In [1]:
import sys
import os
import glob
import librosa
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

In [2]:
def file_to_vector_array(file_path, n_mels=128, frames=5, n_fft=1024, hop_length=512, power=2.0):
    dims = n_mels * frames

    y, sr = librosa.load(file_path, sr=None, mono=True)
    mel_spectrogram = librosa.feature.melspectrogram(y=y,
                                                     sr=sr,
                                                     n_fft=n_fft,
                                                     hop_length=hop_length,
                                                     n_mels=n_mels,
                                                     power=power)

    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max).astype(np.float32)

    vector_array_size = len(log_mel_spectrogram[0, :]) - frames + 1

    if vector_array_size < 1:
        return np.empty((0, dims))

    vector_array = np.zeros((vector_array_size, dims))
    for t in range(frames):
        vector_array[:, n_mels * t: n_mels * (t + 1)] = log_mel_spectrogram[:, t: t + vector_array_size].T

    return vector_array

def list_to_vector_array(file_list, n_mels=128, frames=5, n_fft=1024, hop_length=512, power=2.0):
    all_vectors = []
    for file in file_list:
        vectors = file_to_vector_array(file, n_mels=n_mels, frames=frames, n_fft=n_fft, hop_length=hop_length, power=power)
        all_vectors.append(vectors)

    return np.vstack(all_vectors)

In [3]:
def list_wavs_by_machine_id(train_dir, machine_id):
    training_list_path = os.path.relpath("../dataset/{dir}/*_id_{id}_*.wav".format(dir=train_dir, id=machine_id))
    files = sorted(glob.glob(training_list_path))
    return files

In [4]:
import keras.models
from keras.models import Model
from keras import layers

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

def get_vae(input_dim, latent_dim=20):
    encoder_inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(256, activation="relu")(encoder_inputs)
    x = layers.Dense(128, activation="relu")(x)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var])
    encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    latent_inputs = keras.Input(shape=(latent_dim,))
    x = layers.Dense(128, activation="relu")(latent_inputs)
    x = layers.Dense(256, activation="relu")(x)
    decoder_outputs = layers.Dense(input_dim)(x)
    decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")

    return encoder, decoder

class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(keras.losses.mean_squared_error(data, reconstruction)))
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {"loss": self.total_loss_tracker.result(), "recon": self.reconstruction_loss_tracker.result(), "kl": self.kl_loss_tracker.result()}
    
    def test_step(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(keras.losses.mean_squared_error(data, reconstruction)))
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        total_loss = reconstruction_loss + kl_loss
        
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {"loss": self.total_loss_tracker.result(), "recon": self.reconstruction_loss_tracker.result(), "kl": self.kl_loss_tracker.result()}
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "encoder": self.encoder,
            "decoder": self.decoder,
        })
        return config

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

machine_config = {
    'fan': ['00', '02', '04', '06'],
    'valve': ['00', '02', '04', '06'],
    'pump': ['00', '02', '04', '06'],
    'slider': ['00', '02', '04', '06'],
    'ToyCar': ['01', '02', '03', '04'],
    'ToyConveyor': ['01', '02', '03']
}

vae_results = []

for machine_type, machine_ids in machine_config.items():
    for m_id in machine_ids:
        print(f'--- (VAE) Processing {machine_type} with ID: {m_id} ---')
        train_files = list_wavs_by_machine_id(f'{machine_type}/train', m_id)

        train_files, val_files = train_test_split(train_files, test_size=0.1, random_state=42)

        train_data = list_to_vector_array(train_files)
        val_data = list_to_vector_array(val_files)
        
        scaler = StandardScaler()
        train_data = scaler.fit_transform(train_data)

        val_data = scaler.transform(val_data)

        encoder, decoder = get_vae(128 * 5)
        model = VAE(encoder, decoder)
        model.compile(optimizer="adam")

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ModelCheckpoint(filepath=f'../models/vae/vae_model_{machine_type}_{m_id}.keras', monitor='val_loss', mode='min', save_best_only=True)
        ]

        model.fit(
            train_data,
            epochs=80,
            batch_size=512,
            validation_data=(val_data,),
            callbacks=callbacks
        )

        test_files = list_wavs_by_machine_id(f'{machine_type}/test', m_id)
        y_true = [1 if 'anomaly' in os.path.basename(f) else 0 for f in test_files]

        y_pred = []

        for f in test_files:
            vectors = file_to_vector_array(f)
            vectors = scaler.transform(vectors)
            _, _, z = encoder.predict(vectors, verbose=0)
            recon = decoder.predict(z, verbose=0)
            mse = np.mean(np.square(vectors - recon))
            y_pred.append(mse)

        fpr, tpr, _ = roc_curve(y_true, y_pred)
        roc_auc = auc(fpr, tpr)

        vae_results.append({
            'Machine_Type': machine_type,
            'Machine_ID': m_id,
            'AUC_Score': roc_auc
        })

        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC - {machine_type} with ID {m_id}')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)

        plt.savefig(f'../results/vae/{machine_type}_{m_id}_roc.png')
        plt.close()

--- (VAE) Processing fan with ID: 00 ---
Epoch 1/80
495/495 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - kl: 17.0917 - loss: 221.8778 - recon: 204.7861 - val_kl: 16.2559 - val_loss: 194.8142 - val_recon: 178.5583
Epoch 2/80
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - kl: 16.3468 - loss: 187.9526 - recon: 171.6057 - val_kl: 16.4651 - val_loss: 186.6030 - val_recon: 170.1379
Epoch 3/80
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - kl: 16.3680 - loss: 182.6067 - recon: 166.2388 - val_kl: 16.3206 - val_loss: 183.9577 - val_recon: 167.6371
Epoch 4/80
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - kl: 16.4842 - loss: 180.2271 - recon: 163.7431 - val_kl: 16.5198 - val_loss: 182.3342 - val_recon: 165.8145
Epoch 5/80
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - kl: 16.6643 - loss: 178.8436 - recon: 162.1793 - val_kl: 16.4310 - val_loss: 181.2542 - val_recon: 164.8232
Epoch 6/8

/anaconda/envs/jupyter_env/lib/python3.10/site-packages/keras/src/saving/saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [6]:
results_df = pd.DataFrame(vae_results)
results_df

,Machine_Type,Machine_ID,AUC_Score
0,fan,00,0.557273
1,fan,02,0.789554
2,fan,04,0.625920
3,fan,06,0.865346
4,valve,00,0.758739
5,valve,02,0.749167
6,valve,04,0.775083
7,valve,06,0.624167
8,pump,00,0.678601
9,pump,02,0.628468
